In [ ]:
import mlrose_hiive as mlrh
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [7, 7]

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import plot_precision_recall_curve
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import PrecisionRecallDisplay

random_state = 2020

In [ ]:
X_cred, y_cred = fetch_openml("credit-g", return_X_y=True)

# y_cred[0], cred_mapping = pd.factorize(y_cred[0])
le = LabelEncoder()
y_cred = le.fit_transform(y_cred)

# switch labels around
# good risk: 0
# bad risk: 1
y_cred[y_cred==1] = 3  # Temp
y_cred[y_cred==0] = 1
y_cred[y_cred==3] = 0

# Create holdout test set
X_cred_train, X_cred_test, y_cred_train, y_cred_test = train_test_split(X_cred, y_cred, test_size=0.2,
                                                                        random_state=random_state,
                                                                        stratify=y_cred)
def get_data(data):
    return data["X"], data["y"], data["name"]

# Standardize
scaler_cred = StandardScaler()
scaler_cred.fit(X_cred_train)
X_cred_train = scaler_cred.transform(X_cred_train)
X_cred_test = scaler_cred.transform(X_cred_test)

# Create data sets
cred = {"X": X_cred_train, "y": y_cred_train, "name": "Credit Train"}
cred_test = {"X": X_cred_test, "y": y_cred_test, "name": "Credit Test"}

# Genetic Algorithm Grid Search

In [ ]:
grid_search = {
    "max_iters": [100, 500, 1000, 2500],
    "learning_rate_init": [0.001, 0.1, 0.5],
    "hidden_layers_sizes": [[6, 6]],
    "activation": [mlrh.neural.activation.relu],
    "is_classifier": [True],
    "mutation_prob": [0.1, 0.25, 0.5, 0.7],
    "pop_size": [100, 200, 500, 750, 1000]    
}
X, y, _ = get_data(cred)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=random_state)

runner = mlrh.NNGSRunner(x_train=X_train, y_train=y_train,
                           x_test=X_test, y_test=y_test,
                           experiment_name="full_grid_search",
                           output_directory="nn_genetic_algorithm/",
                           algorithm=mlrh.algorithms.genetic_alg,
                           grid_search_parameters=grid_search,
                           iteration_list = [1, 10, 50, 100, 250, 500, 1000, 2500, 5000, 10000],
                           hidden_layer_sizes=[[6,6]],
                           bias=True,
                           early_stopping=True,
                           clip_max=1,
                           max_attempts=1000,
                           generate_curves=True,
                           seed=random_state,
                           n_jobs=-1
                          )
run_stats, curves, cv_results, best_est = runner.run()

In [ ]:
best_est.best_params_ 